## 1. Environment Preparation

Install Unsloth and updated HuggingFace libraries for Qwen 2.5 support.

In [ ]:
# Install core packages from PyPI (much faster than git installs)
!pip install -q unsloth transformers trl peft accelerate datasets bitsandbytes

# Verify installations
import unsloth
import transformers
import trl
print(f"✓ Unsloth: {unsloth.__version__}")
print(f"✓ Transformers: {transformers.__version__}")
print(f"✓ TRL: {trl.__version__}")
print("Environment ready!")

## 2. Output Configuration & Load Dataset

Define all output paths in one place, then load the Augmentoolkit-generated Marcus Aurelius dataset.


In [ ]:
from pathlib import Path
from datasets import load_dataset, concatenate_datasets
import glob

# ============================================================================
# OUTPUT DIRECTORY STRUCTURE (all outputs organized under one root)
# ============================================================================
OUTPUT_BASE = Path("/home/spark/projects/training/outputs/stoic_qwen25_14b_augmentoolkit")

CHECKPOINTS_DIR = OUTPUT_BASE / "checkpoints"   # Training checkpoints
LORA_DIR        = OUTPUT_BASE / "lora"           # LoRA adapters (lightweight)
MERGED_DIR      = OUTPUT_BASE / "merged"         # Merged 16-bit model (for GGUF)
GGUF_DIR        = OUTPUT_BASE / "gguf"           # GGUF + Modelfile for Ollama

# Create all directories
for d in [CHECKPOINTS_DIR, LORA_DIR, MERGED_DIR, GGUF_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"✓ Output root: {OUTPUT_BASE}")
print(f"  ├── checkpoints/  (training checkpoints)")
print(f"  ├── lora/         (LoRA adapters)")
print(f"  ├── merged/       (merged 16-bit for GGUF)")
print(f"  └── gguf/         (GGUF + Modelfile)")

# ============================================================================
# LOAD DATASET — Only clean ShareGPT files (plain_qa_list.jsonl)
# ============================================================================
# Why plain_qa_list only:
#   - simplified_data_rag has junk system prompt ("does not matter, RAG deprecated")
#   - sft_run/combined_factual_data duplicates these with overlap
#   - inferred_facts, representation_variation, pretraining = raw {text}, not ShareGPT
#   - rag_data, rag_source_data = {segments} or {question}, not ShareGPT
#   - generic_sft = all empty (0 lines)
# ============================================================================
base_path = "/home/spark/projects/augmentoolkit/outputs/marcus_aurelius_dataset"

# Only load plain_qa_list.jsonl from factual_sft directories (clean system prompts, ShareGPT format)
jsonl_files = sorted(glob.glob(f"{base_path}/factual_sft_stoics_*/plain_qa_list.jsonl"))

print(f"Found {len(jsonl_files)} clean JSONL files in {base_path}")

# Load all JSONL files
datasets_list = []
total_examples = 0

for file_path in jsonl_files:
    ds = load_dataset("json", data_files=file_path, split="train")
    if len(ds) > 0:
        datasets_list.append(ds)
        total_examples += len(ds)
        rel_path = file_path.replace(base_path + "/", "")
        print(f"  Loaded {len(ds):>4} examples from {rel_path}")

dataset = concatenate_datasets(datasets_list)

print(f"\n✓ Total dataset loaded: {len(dataset)} examples")
print(f"✓ Columns: {dataset.column_names}")
print(f"\n--- First Example (first 800 chars) ---")
import json
print(json.dumps(dataset[0], indent=2)[:800])

# Shuffle dataset for better training
dataset = dataset.shuffle(seed=42)
print(f"\n✓ Dataset shuffled and ready for training")

## 3. Load Model & Tokenizer with Unsloth

Load Qwen 2.5-14B-Instruct model with 4-bit quantization (14B requires quantization to fit in GPU memory).

In [ ]:
from unsloth import FastLanguageModel
import torch
from transformers import BitsAndBytesConfig

model_name = "unsloth/Qwen2.5-14B-Instruct"
max_seq_length = 2048

# Configure 4-bit quantization (essential for 14B model to fit in GPU memory)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    llm_int8_enable_fp32_cpu_offload=True
)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
    quantization_config=bnb_config,
    device_map={"": 0}  # Force all on GPU 0
)

# Qwen 2.5 uses <|endoftext|> as eos; set pad token accordingly
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"✓ Model loaded: {model_name}")
print(f"✓ Precision: 4-bit NF4 quantized")
print(f"✓ Tokenizer configured (ChatML format)")
print(f"✓ Max sequence length: {max_seq_length}")

In [ ]:
# Format ShareGPT dataset for Qwen 2.5 ChatML template
def format_instruct(example):
    # Augmentoolkit uses ShareGPT format: "conversations" field with from/value
    # Convert to Qwen 2.5 ChatML format (role/content)
    # Qwen 2.5 uses: <|im_start|>role\ncontent<|im_end|>
    messages = []
    for turn in example["conversations"]:
        # ShareGPT uses "from": "system"/"human"/"gpt"
        # Qwen 2.5 uses "role": "system"/"user"/"assistant"
        if turn["from"] == "system":
            messages.append({"role": "system", "content": turn["value"]})
        elif turn["from"] == "human":
            messages.append({"role": "user", "content": turn["value"]})
        elif turn["from"] == "gpt":
            messages.append({"role": "assistant", "content": turn["value"]})
    
    text = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=False
    )
    return {"text": text}

# Format and remove old columns
dataset = dataset.map(format_instruct, remove_columns=dataset.column_names)

print(f"✓ Dataset formatted: {len(dataset)} examples")
print(f"\n--- Sample formatted text (first 500 chars) ---")
print(dataset[0]['text'][:500])

## 4. Add LoRA Adapters

Configure LoRA for efficient fine-tuning with attention and MLP projection layers.


In [ ]:
from peft import LoraConfig

# Add LoRA adapters using Unsloth's method
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    max_seq_length=max_seq_length
)

print("✓ LoRA adapters added successfully")
print(f"✓ Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


## 5. Trainer Setup & Training

**Augmentoolkit Dataset:**
- First-person Stoic responses generated from Marcus Aurelius' Meditations
- ~280 examples from `plain_qa_list.jsonl` across 40 files (5 SFT types × 8 shards)
- Types: followup, hallucination, negative, openended, vague
- Designed to teach the model to speak AS a Stoic, not about Stoicism

**Training Configuration:**
- Cosine LR decay for smooth convergence
- 3 epochs for this dataset size
- System prompts already embedded in Augmentoolkit data

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Dynamic training configuration based on actual dataset size
# Reduced batch size for 14B model (more VRAM usage than 7B)
batch_size = 1
grad_accum = 8
effective_batch_size = batch_size * grad_accum

# Calculate steps for 3 epochs (safer to avoid overfitting)
steps_per_epoch = len(dataset) // effective_batch_size
target_epochs = 3
max_steps = steps_per_epoch * target_epochs

# Set warmup to ~10% of total steps, save every epoch
warmup_steps = max(1, max_steps // 10)
save_steps = steps_per_epoch

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=grad_accum,
        warmup_steps=warmup_steps,
        max_steps=max_steps,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir=str(CHECKPOINTS_DIR),
        report_to="none",
        save_strategy="steps",
        save_steps=save_steps,
    )
)

print("✓ Trainer configured for Qwen 2.5-14B + Augmentoolkit dataset")
print(f"✓ Dataset size: {len(dataset)} conversations")
print(f"✓ Effective batch size: {effective_batch_size} (batch={batch_size} × grad_accum={grad_accum})")
print(f"✓ Steps per epoch: {steps_per_epoch}")
print(f"✓ Total epochs: {target_epochs}")
print(f"✓ Total steps: {max_steps}")
print(f"✓ Warmup steps: {warmup_steps}")
print(f"✓ Save every: {save_steps} steps (every epoch)")
print(f"✓ Checkpoints: {CHECKPOINTS_DIR}")

In [ ]:
# Start training
trainer.train()

In [ ]:
# Save LoRA adapters immediately (lightweight, fast ~50-100MB)
model.save_pretrained(str(LORA_DIR))
tokenizer.save_pretrained(str(LORA_DIR))

print(f"✓ LoRA adapters saved to {LORA_DIR}")
print("  (Can reload later with PeftModel.from_pretrained if merged save fails)")

## 7. Save Model & Inference

Save the fine-tuned model (16-bit merged for clean GGUF conversion) and test inference with a Stoic question.

In [ ]:
# Save merged model in 16-bit (required for clean GGUF conversion)
# Why 16-bit: saving as 4-bit then converting to GGUF would double-quantize
# (4-bit NF4 → lossy dequant to fp16 → re-quantize to Q4_K_M = quality loss)
# Saving 16-bit first means GGUF quantization only happens once.
model.save_pretrained_merged(str(MERGED_DIR), tokenizer, save_method="merged_16bit")

print(f"✓ Merged 16-bit model saved to {MERGED_DIR}")
print(f"  (16-bit is the source for single-pass GGUF quantization)")

In [ ]:
# Prepare model for inference
FastLanguageModel.for_inference(model)

# Test inference with a Stoic question
test_prompt = "What troubles me today is the judgment of others. How should I view this?"

inputs = tokenizer.apply_chat_template(
    [{"role": "user", "content": test_prompt}],
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=256,
    temperature=0.7, 
    top_p=0.9,
    repetition_penalty=1.1
)
response = tokenizer.decode(outputs[0], skip_special_tokens=False)

print("\n=== RAW FULL OUTPUT (with tags) ===")
print(response)
print("\n=== PARSED OUTPUT ===")
print(f"User: {test_prompt}")
# Qwen 2.5 uses ChatML: parse after <|im_start|>assistant
assistant_tag = "<|im_start|>assistant\n"
if assistant_tag in response:
    parsed = response.split(assistant_tag)[-1].replace("<|im_end|>", "").strip()
else:
    parsed = response
print(f"\nAssistant: {parsed}")

## Notes

### Dataset Quality
This notebook loads only `plain_qa_list.jsonl` files from the Augmentoolkit pipeline output — ~280 clean ShareGPT conversations with proper first-person Stoic system prompts. Excluded files:
- `simplified_data_rag` — deprecated system prompt ("does not matter, RAG is deprecated")
- `sft_run/combined_factual_data` — partial duplicates of plain_qa_list
- `inferred_facts`, `representation_variation`, `pretraining_run` — raw `{text}` format, not ShareGPT
- `rag_data`, `rag_source_data` — `{segments}` or `{question}` format, not ShareGPT
- `generic_sft` — all 6 files are empty (0 lines)

### GGUF Pipeline
Merged model is saved in 16-bit to avoid double quantization. The GGUF pipeline quantizes once: 16-bit → Q4_K_M (or your chosen quant type).

### Next Steps
- For Epictetus dataset: Run Augmentoolkit with Enchiridion/Discourses
- Merge datasets: Combine Marcus + Epictetus for broader Stoic knowledge
- Evaluate first-person quality: Test if model says "I practice..." vs "Stoics believe..."

## 8. Convert to GGUF for Ollama

Convert the fine-tuned model to GGUF format for use with Ollama. Supports both full-precision (fp32) and quantized formats (q4, q8, etc.) for different size/quality trade-offs.

**Quantization Options:**
- `None`: Full precision (fp32) - maximum quality, largest size
- `"q4"`: 4-bit (Q4_0) - highest compression, fastest inference
- `"q8"`: 8-bit (Q8_0) - best balance of size/speed/quality ⭐ **Recommended**
- `"q4_k"`: 4-bit K-quant (Q4_K_M) - better quality than q4
- `"q5_k"`: 5-bit K-quant (Q5_K_M) - excellent quality, moderate size
- `"q6_k"`: 6-bit K-quant (Q6_K) - near-lossless quality

In [ ]:
import subprocess
import sys

# ============================================================================
# GGUF CONVERSION CONFIGURATION
# ============================================================================
LLAMA_CPP_PATH = Path("/home/spark/resources/llama.cpp")
SOURCE_MODEL_DIR = MERGED_DIR  # Use the merged model from earlier

# Set to "q8" for best balance, "q4_k" for production, or None for full precision
QUANTIZATION_TYPE = "q4_k"  # Options: None, "q4", "q8", "q4_k", "q5_k", "q6_k", "fp16"

# ============================================================================

print(f"🔧 GGUF Conversion for Ollama")
print(f"   Source model: {SOURCE_MODEL_DIR}")
print(f"   Quantization: {QUANTIZATION_TYPE or 'Full Precision (fp32)'}")
print(f"   Output: {GGUF_DIR}")

# Verify llama.cpp exists
if not LLAMA_CPP_PATH.exists():
    raise FileNotFoundError(
        f"❌ llama.cpp not found at {LLAMA_CPP_PATH}\n"
        f"   This DGX uses a shared resources folder.\n"
        f"   Please clone it: git clone https://github.com/ggerganov/llama.cpp {LLAMA_CPP_PATH}\n"
        f"   Then build it: cd {LLAMA_CPP_PATH} && cmake -B build -DLLAMA_CURL=OFF && cmake --build build -j$(nproc)"
    )

print(f"✓ llama.cpp found at {LLAMA_CPP_PATH}")

In [ ]:
if QUANTIZATION_TYPE is None:
    # ========================================================================
    # Full Precision (fp32) - No Quantization
    # ========================================================================
    print("\n[Step 1/1] Converting to full-precision GGUF (fp32)...")
    
    FINAL_GGUF = GGUF_DIR / "stoic-qwen25-14b-fp32.gguf"
    
    subprocess.run([
        sys.executable,
        str(LLAMA_CPP_PATH / "convert_hf_to_gguf.py"),
        str(SOURCE_MODEL_DIR),
        "--outfile",
        str(FINAL_GGUF),
        "--outtype",
        "f32",
    ], check=True)
    
    print(f"   ✅ Full-precision GGUF: {FINAL_GGUF.name}")
    
else:
    # ========================================================================
    # Quantized Conversion (2-step process)
    # ========================================================================
    print("\n[Step 1/2] Converting to fp16 GGUF (pre-quantization)...")
    
    TEMP_GGUF = GGUF_DIR / "temp-fp16.gguf"
    
    subprocess.run([
        sys.executable,
        str(LLAMA_CPP_PATH / "convert_hf_to_gguf.py"),
        str(SOURCE_MODEL_DIR),
        "--outfile",
        str(TEMP_GGUF),
        "--outtype",
        "f16",
    ], check=True)
    
    print(f"   ✅ fp16 GGUF created: {TEMP_GGUF.name}")
    
    # Step 2: Quantize the fp16 GGUF
    print(f"\n[Step 2/2] Quantizing to {QUANTIZATION_TYPE}...")
    
    FINAL_GGUF = GGUF_DIR / f"stoic-qwen25-14b-{QUANTIZATION_TYPE}.gguf"
    
    quant_map = {
        "q4": "Q4_0",
        "q8": "Q8_0",
        "q4_k": "Q4_K_M",
        "q5_k": "Q5_K_M",
        "q6_k": "Q6_K",
        "fp16": "F16",
    }
    llama_quant_type = quant_map.get(QUANTIZATION_TYPE, QUANTIZATION_TYPE.upper())
    
    quantize_binary = LLAMA_CPP_PATH / "build" / "bin" / "llama-quantize"
    
    if not quantize_binary.exists():
        raise FileNotFoundError(
            f"❌ llama-quantize binary not found at {quantize_binary}\n"
            f"   Please build llama.cpp:\n"
            f"   cd {LLAMA_CPP_PATH} && cmake -B build -DLLAMA_CURL=OFF && cmake --build build --config Release -j$(nproc)"
        )
    
    subprocess.run([
        str(quantize_binary),
        str(TEMP_GGUF),
        str(FINAL_GGUF),
        llama_quant_type,
    ], check=True)
    
    # Clean up temp file
    TEMP_GGUF.unlink()
    print(f"   ✅ Quantized GGUF: {FINAL_GGUF.name} ({llama_quant_type})")

print(f"\n✅ GGUF conversion complete: {FINAL_GGUF}")

In [ ]:
# Create Modelfile for Ollama
print("\n📝 Creating Modelfile for Open WebUI...")

MODELFILE_PATH = GGUF_DIR / "Modelfile"

# Qwen 2.5 uses ChatML template
modelfile_content = f"""FROM ./{FINAL_GGUF.name}

TEMPLATE \"\"\"<|im_start|>system
{{{{ .System }}}}<|im_end|>
<|im_start|>user
{{{{ .Prompt }}}}<|im_end|>
<|im_start|>assistant
\"\"\"

PARAMETER stop "<|im_end|>"
PARAMETER stop "<|im_start|>"
"""

MODELFILE_PATH.write_text(modelfile_content, encoding="utf-8")

print(f"✅ Modelfile created: {MODELFILE_PATH}")
print(f"\n🚀 Ready for Open WebUI!")
print(f"\nTo import into Ollama:")
print(f"   cd {GGUF_DIR}")
print(f"   ollama create stoic-qwen25-14b -f Modelfile")
print(f"\nThen configure in Open WebUI:")
print(f"   - Select 'stoic-qwen25-14b' as base model")
print(f"   - Add custom system prompt")
print(f"   - Set temperature, top_p, etc. via UI sliders")